# SIMPLIcity CNN Image Classification Experiment
### Course: Neuro Computing (PENS) - Case Study: Image Classification with SIMPLIcity Dataset

This notebook provides a complete, reproducible workflow to train and evaluate CNN models on Google Colab or Kaggle with GPU acceleration.

**Models Evaluated:**
1. `baseline_custom`: 2-layer CNN baseline as defined in Slide 6 of the assignment.
2. `custom_scratch`: Custom CNN designed from scratch with Batch Normalization, Dropout, and Adaptive Pooling.
3. `resnet18`: Transfer learning benchmark using ImageNet pretrained backbone.

## 1. Verify GPU Acceleration
Make sure runtime is set to **GPU** (`Runtime -> Change runtime type -> T4 GPU` in Colab).

In [ ]:
!nvidia-smi
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:     {torch.cuda.get_device_name(0)}")

## 2. Setup Environment & Repository
If running on Google Colab, you can clone your GitHub repository or upload the codebase zip.

In [ ]:
# Install project in editable mode
%pip install -e .
# Or install required dependencies
%pip install -r requirements.txt

## 3. Dataset Setup
If you packaged your dataset using `python scripts/package_dataset.py`, upload `simplicity_dataset.zip` here and run the cell below.

In [ ]:
import os
import zipfile

zip_path = "simplicity_dataset.zip"
target_dir = "Dataset/SIMPLIcity/SIMPLIcity Dataset"

if os.path.exists(zip_path) and not os.path.exists(os.path.join(target_dir, "train")):
    print(f"Extracting {zip_path} to {target_dir}...")
    os.makedirs(target_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(target_dir)
    print("Extraction complete!")
else:
    print("Dataset directory already verified or zip not found in current folder.")

## 4. Run Dataset Audit
Verify integrity of images and distribution across the 10 classes in train, validation, and test splits.

In [ ]:
!python scripts/audit_data.py

## 5. Experiment 1: Baseline Custom CNN (Slide 6 Assignment)
Architecture:
- Conv(64, 3x3) + ReLU + MaxPool(2x2)
- Conv(128, 3x3) + ReLU + MaxPool(2x2)
- Linear(131072, 128) + ReLU
- Linear(128, 10)

In [ ]:
!python scripts/train.py \
    --model baseline_custom \
    --epochs 30 \
    --batch-size 32 \
    --lr 0.001 \
    --output-dir outputs/baseline_custom

### Display Baseline Curves & Confusion Matrix

In [ ]:
from IPython.display import Image, display

print("=== Baseline CNN Gradient Descent Curves ===")
display(Image("outputs/baseline_custom/gradient_descent_summary.png"))

print("=== Baseline CNN Test Confusion Matrix ===")
display(Image("outputs/baseline_custom/evaluation/confusion_matrix.png"))

## 6. Experiment 2: Regularized Custom CNN from Scratch
Architecture:
- 3 Conv stages with Batch Normalization & Dropout
- Adaptive Average Pooling (512 features) to mitigate parameter explosion
- Fully Connected with Dropout (0.4)

In [ ]:
!python scripts/train.py \
    --model custom_scratch \
    --epochs 35 \
    --batch-size 32 \
    --lr 0.001 \
    --output-dir outputs/custom_scratch

### Display Regularized Custom CNN Curves & Confusion Matrix

In [ ]:
print("=== Regularized Custom CNN Gradient Descent Curves ===")
display(Image("outputs/custom_scratch/gradient_descent_summary.png"))

print("=== Regularized Custom CNN Test Confusion Matrix ===")
display(Image("outputs/custom_scratch/evaluation/confusion_matrix.png"))

## 7. Experiment 3: Transfer Learning Benchmark (ResNet18)

In [ ]:
!python scripts/train.py \
    --model resnet18 \
    --epochs 20 \
    --batch-size 32 \
    --lr 0.0003 \
    --output-dir outputs/resnet18

### Display ResNet18 Curves & Confusion Matrix

In [ ]:
print("=== ResNet18 Gradient Descent Curves ===")
display(Image("outputs/resnet18/gradient_descent_summary.png"))

print("=== ResNet18 Test Confusion Matrix ===")
display(Image("outputs/resnet18/evaluation/confusion_matrix.png"))

## 8. Comparative Results Summary
Generate comparative analysis table across all experiments.

In [ ]:
import json

import pandas as pd

models = ["baseline_custom", "custom_scratch", "resnet18"]
rows = []

for m in models:
    report_p = f"outputs/{m}/evaluation/classification_report.json"
    hist_p = f"outputs/{m}/history.json"
    if os.path.exists(report_p) and os.path.exists(hist_p):
        with open(report_p) as f:
            rep = json.load(f)
        with open(hist_p) as f:
            hist = json.load(f)
        rows.append(
            {
                "Model": m,
                "Best Val Acc (%)": round(max(hist["val_acc"]), 2),
                "Test Accuracy (%)": round(rep["accuracy"] * 100, 2),
                "Macro F1": round(rep["macro avg"]["f1-score"], 4),
                "Macro Precision": round(rep["macro avg"]["precision"], 4),
                "Macro Recall": round(rep["macro avg"]["recall"], 4),
            }
        )

if rows:
    df_summary = pd.DataFrame(rows)
    display(df_summary)
else:
    print("Run experiments above to populate the summary table.")